In [12]:
%pip install tensorflow_hub tensorflow-text --user

  Using cached tensorflow_hub-0.16.1-py2.py3-none-any.whl.metadata (1.3 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow-text (from versions: none)

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow-text


In [1]:
import os
import sys
import nltk
import torch

# 1. Setup local folders for data (Avoids permission/Admin errors)
current_dir = os.getcwd()
nltk_dir = os.path.join(current_dir, 'nltk_data')
tf_cache_dir = os.path.join(current_dir, 'tfhub_modules')

for folder in [nltk_dir, tf_cache_dir]:
    if not os.path.exists(folder):
        os.makedirs(folder)

# 2. Configure Environment Paths
os.environ['TFHUB_CACHE_DIR'] = tf_cache_dir
if nltk_dir not in nltk.data.path:
    nltk.data.path.append(nltk_dir)

# 3. Quick Download NLTK resources
print("📦 Fetching NLTK resources...")
nltk.download('averaged_perceptron_tagger_eng', download_dir=nltk_dir, quiet=True)
nltk.download('punkt_tab', download_dir=nltk_dir, quiet=True)
nltk.download('universal_tagset', download_dir=nltk_dir, quiet=True)
nltk.download('wordnet', download_dir=nltk_dir, quiet=True)

# 4. Memory Cleanup for RTX A4000
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Environment successfully patched for Lab GPU! Now run your attack cell.")

📦 Fetching NLTK resources...
✅ Environment successfully patched for Lab GPU! Now run your attack cell.


In [2]:
import os
import torch
import pandas as pd
import nltk
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.datasets import Dataset
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.constraints.grammaticality import PartOfSpeech
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import WordSwapEmbedding
from textattack.goal_functions import UntargetedClassification
from textattack import Attack, AttackArgs, Attacker

# --- STEP 1: NLTK SETUP ---
download_dir = os.path.join(os.getcwd(), 'nltk_data')
if not os.path.exists(download_dir): os.makedirs(download_dir)
nltk.download('averaged_perceptron_tagger_eng', download_dir=download_dir, quiet=True)
nltk.download('punkt_tab', download_dir=download_dir, quiet=True)
nltk.download('universal_tagset', download_dir=download_dir, quiet=True)
if download_dir not in nltk.data.path: nltk.data.path.append(download_dir)

# --- STEP 2: DEVICE & DATA ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Lab GPU: {torch.cuda.get_device_name(0)}")

df = pd.read_csv("adversarial_test_data_2000.csv").head(100)
dataset = Dataset(list(zip(df['text'].astype(str), df['label'])))

# --- STEP 3: MODEL LOADING ---
model_path = "./final-roberta-hc3"
model = AutoModelForSequenceClassification.from_pretrained(model_path, torch_dtype=torch.float16).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

# --- STEP 4: CUSTOM PYTORCH-ONLY TEXTFOOLER ---
# Instead of Universal Sentence Encoder, we use Word Embedding Distance (PyTorch compatible)
goal_function = UntargetedClassification(model_wrapper)
transformation = WordSwapEmbedding(max_candidates=50)

constraints = [
    # 1. Semantic Constraint: Ensures synonyms are contextually similar
    WordEmbeddingDistance(min_cos_sim=0.5),
    # 2. Grammatical Constraint: Ensures Part-of-Speech is preserved
    PartOfSpeech(tagger_type='nltk', tagset='universal', allow_verb_noun_swap=True)
]

search_method = GreedyWordSwapWIR(wir_method="delete")
attack = Attack(goal_function, constraints, transformation, search_method)

# --- STEP 5: EXECUTION ---
attack_args = AttackArgs(
    num_examples=50, 
    log_to_csv="adversarial_results_pytorch_only.csv",
    disable_stdout=False
)

attacker = Attacker(attack, dataset, attack_args)
print("🔥 Starting PyTorch-Optimized Attack (No TensorFlow needed)...")
attacker.attack_dataset()

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


🚀 Lab GPU: NVIDIA RTX A4000


textattack: Unknown if model of class <class 'transformers.models.roberta.modeling_roberta.RobertaForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\textattack\shared\word_embeddings.py:296: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  mse_dist_mat = pickle.load(f)
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\textattack\shared\word_embeddings.py:298: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  cos_sim_mat = pickle.load(f)
textattack: Logging to CSV at path adversarial_results_pytorch_only.csv


🔥 Starting PyTorch-Optimized Attack (No TensorFlow needed)...
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 0 / 1 / 0 / 1:   2%|▏         | 1/50 [00:20<16:25, 20.11s/it]

...

--------------------------------------------- Result 48 ---------------------------------------------
[[1 (100%)]] --> [[[FAILED]]]

["The colonists in the American colonies in the 1700s did not dislike the British as a whole, but they did have a number of grievances with the British government and the way they were being treated. One of the main issues was that the colonists felt that they were not being treated as equals by the British and were being unfairly taxed. They also resented the fact that they had no representation in the British government and were being dictated to by officials who had no understanding of their needs or concerns. \nIn addition to these political issues, the colonists also had cultural and economic differences with the British. Many of the colonists had come to the New World to escape religious persecution and sought to establish their own way of life. They also wanted to be able to control their own economic destiny and believed that the British were tryi

[Succeeded / Failed / Skipped / Total] 3 / 46 / 0 / 49:  98%|█████████▊| 49/50 [22:14<00:27, 27.24s/it]

--------------------------------------------- Result 49 ---------------------------------------------
[[0 (100%)]] --> [[[FAILED]]]

['There \'s a concept called the " Hayflick limit " , which basically says , a cell can only divide so many times before it loses the ability to . They believe it occurs on the DNA level , a piece is lost each replication , and when enough has been lost , the cell ca n\'t reproduce again . All living creatures are constantly breaking down , and repairing themselves with new cells . When they ca n\'t fix the broken parts , they begin to fail , and death is on the way . The rate the break down / repair happens , and the amount of times a cell can be replaced , varies by animal , resulting in drastically different lifespans . It \'s worth mentioning , that in nature , animals ( including humans ) rarely live to this point . Predators , accidents , sickness , and excessive wear and tear ( teeth worn to the point they ca n\'t eat , for instance ) are generally

[Succeeded / Failed / Skipped / Total] 3 / 47 / 0 / 50: 100%|██████████| 50/50 [23:37<00:00, 28.36s/it]

--------------------------------------------- Result 50 ---------------------------------------------
[[0 (100%)]] --> [[[FAILED]]]

["Most of the time , any location wo nt do . They want a specific look , so they build a set to fit that look . Or there s nothing like it nearby , and it 's cheaper than paying everyone to live somewhere else for the length of the shoot . Also , many public places simply ca n't be close off that long . If you 're filming a movie over the course of a year , you might be able to shut down an area for a few days of shooting , but if you need that space often you 're not going to be able to shut down a park or roads every few days . Also , if you 're filming in a public area it 's hard to stop the public from wandering in or watching from as close as possible . also , they can build the set without certain walls or in certain ways so camera angles that are n't possible in real places you can suddenly do . And if you have to destroy something ..... or have so

In [2]:
import os
import torch
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.datasets import Dataset
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import WordSwapEmbedding
from textattack.goal_functions import UntargetedClassification
from textattack import Attack, AttackArgs, Attacker

# --- 1. DEVICE & DATA ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Lab GPU: {torch.cuda.get_device_name(0)}")

# Let's attack 200 examples this time for a larger sample size
df = pd.read_csv("adversarial_test_data_2000.csv").head(200)
dataset = Dataset(list(zip(df['text'].astype(str), df['label'])))

# --- 2. MODEL LOADING ---
model_path = "./final-roberta-hc3"
model = AutoModelForSequenceClassification.from_pretrained(model_path, torch_dtype=torch.float16).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

# --- 3. THE AGGRESSIVE TEXTFOOLER ---
goal_function = UntargetedClassification(model_wrapper)

# MOD 1: Increase max_candidates from 50 to 100. (The attacker has twice as many words to choose from)
transformation = WordSwapEmbedding(max_candidates=100)

constraints = [
    # MOD 2: Lower the similarity threshold from 0.5 to 0.2. (Allows weirder, less exact synonyms)
    WordEmbeddingDistance(min_cos_sim=0.2)
    
    # MOD 3: We completely removed the PartOfSpeech constraint. 
    # The attacker can now swap nouns for verbs or adjectives, easily breaking the model's logic.
]

# Using the standard Word Importance Ranking (WIR) search method
search_method = GreedyWordSwapWIR(wir_method="delete")
attack = Attack(goal_function, constraints, transformation, search_method)

# --- 4. EXECUTION ---
attack_args = AttackArgs(
    num_examples=200, 
    log_to_csv="adversarial_results_aggressive_attack.csv",
    disable_stdout=False
)

attacker = Attacker(attack, dataset, attack_args)
print("🔥 Starting Aggressive Attack (Loose Constraints)...")
attacker.attack_dataset()

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


🚀 Lab GPU: NVIDIA RTX A4000


textattack: Unknown if model of class <class 'transformers.models.roberta.modeling_roberta.RobertaForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\textattack\shared\word_embeddings.py:296: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  mse_dist_mat = pickle.load(f)
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\textattack\shared\word_embeddings.py:298: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  cos_sim_mat = pickle.load(f)
textattack: Logging to CSV at path adversarial_results_aggressive_attack.csv


🔥 Starting Aggressive Attack (Loose Constraints)...
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  100
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.2
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 1 / 0 / 0 / 1:   0%|          | 1/200 [00:46<2:35:20, 46.84s/it]

...

--------------------------------------------- Result 198 ---------------------------------------------
[[1 (100%)]] --> [[[FAILED]]]

["It's important to understand that the situation in Ukraine is complex and there are many different factors at play. Russia has a long history with Ukraine, and the two countries have close cultural, economic, and political ties. \nOne factor that has contributed to the current crisis is Russia's concern about the influence of Western countries, particularly the United States, in Ukraine. Russia has historically seen Ukraine as being within its sphere of influence and has been wary of any efforts by Western countries to gain influence there. \nAnother factor is the issue of Crimea, which is a peninsula in the Black Sea that was part of Ukraine but was annexed by Russia in 2014. This annexation was not recognized by most of the international community and was met with condemnation and economic sanctions. \nOverall, it's difficult to say exactly what Russ

[Succeeded / Failed / Skipped / Total] 21 / 178 / 0 / 199: 100%|█████████▉| 199/200 [6:23:04<01:55, 115.50s/it]

--------------------------------------------- Result 199 ---------------------------------------------
[[0 (100%)]] --> [[[FAILED]]]

["Everything . Diplomats can not even be * detained * in most cases ( although they are liable for parking tickets ) . A diplomat can literally shoot a man in broad daylight and your host country would have to let you go ( see the murder of English Policewoman Yvonne Fletcher by a Libyan embassy worker in 1984 ) . All this results in a lot of spy vs spy hijinks as a suspicious number of them have diplomatic passports . But , the host country is allowed to declare you persona non grata and kick the entire embassy out of the country for it . And as having a criminal in your ranks is a major embarrassment on the world stage , most country would gladly hang the criminal out to dry ( unless of course , he 's one of their spies ) . And sometimes , the punishment that would await that man back home is worse that what the host country can dish out ."
 'My dad ( 

[Succeeded / Failed / Skipped / Total] 21 / 179 / 0 / 200: 100%|██████████| 200/200 [6:24:07<00:00, 115.24s/it]

--------------------------------------------- Result 200 ---------------------------------------------
[[1 (100%)]] --> [[[FAILED]]]

["There are a few reasons why gas prices in the US have recently become lower. One reason is that there has been a decrease in the demand for gasoline due to the COVID-19 pandemic and the resulting economic downturn. When people are driving less and there is less demand for gasoline, the price tends to go down. \nAnother reason is that there is currently a surplus of oil and gasoline, which means there is more of these products available than people are currently using. This can also cause the price to go down. \nIt's hard to predict exactly how long gas prices will stay low, as they are influenced by a number of different factors. These can include things like the global demand for oil, the price of oil, and the cost of transporting gasoline to different parts of the country. \nIn general, it's important to remember that the price of gasoline can fluctu